# Data cleaning & manipulation .ipynb

## Libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from category_encoders import TargetEncoder
import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))
from src.data_cleaning_and_manipulations import impute_by_agency_line_hour

## Imports

In [2]:
df = pd.read_csv(r'../data/model_datasets/train_df.csv',encoding='utf-8-sig')

## 1. Data types

### 1.1. Date & time

In [3]:
## Date
df['date'] = pd.to_datetime(df['date'])

### Departure and arrival attributes
df['departure_time_planned'] = pd.to_datetime(
    df['date'].astype(str) + ' ' + df['departure_time_planned'],
    format='%Y-%m-%d %H:%M:%S'
)

df['arrival_time_planned'] = pd.to_datetime(
    df['date'].astype(str) + ' ' + df['arrival_time_planned'],
    format='%Y-%m-%d %H:%M:%S'
)

### 1.2. Numeric

In [4]:
## Hour
df = df.rename(columns={'hour_rounded': 'full_hour'})
print(df['full_hour'].dtype)

## Line_num
df['line_num'] = pd.to_numeric(df['line_num'], errors='coerce').astype('Int64')
print(df['line_num'].dtype)

int64
Int64


### 1.3. Categorial

In [5]:
# day - categorical with order
day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
df['day'] = pd.Categorical(df['day'], categories=day_order, ordered=True)
print(df['day'].dtype)

str_cols = ['line_name', 'alternative', 'agency_name', 'origin_city', 
            'origin_station', 'destination_city', 'destination_station', 'route_type']

df[str_cols] = df[str_cols].astype(str)

df['route_type'] = df['route_type'].str.strip()

category


## 2. Missing values handling

### 2.1. Duration & Speed (including 'Target' variable)

In [6]:
selected_cols = [
    'departure_time_planned',
    'arrival_time_planned',
    'duration_min_planned',
    'duration_min_actual',
    'speed_kmh_planned',
    'speed_kmh_actual',
    'duration_difference_min'
]



mask = df['duration_min_planned'].isna()
df.loc[mask, 'duration_min_planned'] = (
    (df.loc[mask, 'arrival_time_planned'] - df.loc[mask, 'departure_time_planned'])
    .dt.total_seconds() / 60
)

print(f"Missing values filled: {mask.sum()}")
(df[selected_cols].head())
mask2 = df['duration_difference_min'].isna()
df.loc[mask2, 'duration_difference_min'] = (
    df.loc[mask2, 'duration_min_actual'] - df.loc[mask2, 'duration_min_planned']
)

print(f"Missing values filled: {mask2.sum()}")
(df[selected_cols].head())

mask3 = df['speed_kmh_planned'].isna()
df.loc[mask3, 'speed_kmh_planned'] = (
    (df.loc[mask3, 'route_length'] / 1000) / (df.loc[mask3, 'duration_min_planned'] / 60)
)

print(f"Missing values filled: {mask3.sum()}")
print(df[selected_cols].head())

print(f"duration_min_planned < 0: {(df['duration_min_planned'] < 0).sum():,}")
print(f"speed_kmh_planned < 0:    {(df['speed_kmh_planned'] < 0).sum():,}")

df = df[(df['duration_min_planned'] >= 0) & (df['speed_kmh_planned'] >= 0)]

print(f"\nRows after drop: {len(df):,}")

missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percent': df.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values(by='missing_percent', ascending=False)

missing_summary

Missing values filled: 11888
Missing values filled: 11916
Missing values filled: 11888
  departure_time_planned arrival_time_planned  duration_min_planned  \
0    2024-04-07 15:00:00  2024-04-07 15:48:50             48.833333   
1    2024-04-07 14:00:00  2024-04-07 14:49:59             49.983333   
2    2024-04-11 13:10:00  2024-04-11 14:09:59             60.000000   
3    2024-04-09 15:25:00  2024-04-09 16:02:13             37.000000   
4    2024-04-12 07:30:00  2024-04-12 08:27:08             57.000000   

   duration_min_actual  speed_kmh_planned  speed_kmh_actual  \
0                 72.0          29.643538              17.7   
1                 68.0          22.021075              13.9   
2                 81.0          19.100000              14.1   
3                 58.0          39.800000              25.4   
4                 73.0          18.000000              14.0   

   duration_difference_min  
0                23.166667  
1                18.016667  
2                21.

,missing_count,missing_percent
Avg_Passengers_Per_Bus,1294,2.340345
Total_Passengers,1294,2.340345
line_num,126,0.227885
duration_min_actual,78,0.141072
duration_difference_min,78,0.141072
speed_kmh_actual,78,0.141072
curvity,24,0.043407
perc_within_pt_route,24,0.043407
route_length,24,0.043407
length_in_buffer_m,24,0.043407


### 2.2. Passengrs columns

In [7]:
for col in ['Total_Passengers']:
    df[col] = df.groupby(['route_id', 'direction', 'day', 'full_hour'])[col].transform(
        lambda x: x.fillna(x.median())
    )
    print(f"{col} missing after fill: {df[col].isna().sum()}")


# שלב 2 - groupby פחות ספציפי
for col in ['Total_Passengers']:
    df[col] = df.groupby(['route_id', 'direction', 'day'])[col].transform(
        lambda x: x.fillna(x.median())
    )
    print(f"{col} missing after step 2: {df[col].isna().sum()}")

# שלב 3 - אם עדיין חסר, לפי route_id בלבד
for col in ['Total_Passengers']:
    df[col] = df.groupby(['route_id'])[col].transform(
        lambda x: x.fillna(x.median())
    )
    print(f"{col} missing after step 3: {df[col].isna().sum()}")

# שלב 4 - אם עדיין חסר, מלא עם חציון כללי
for col in ['Total_Passengers']:
    df[col] = df[col].fillna(df[col].median())
    print(f"{col} missing after step 4: {df[col].isna().sum()}")


missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percent': df.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values(by='missing_percent', ascending=False)

missing_summary

C:\Users\shaha\AppData\Local\Temp\ipykernel_13576\2633105291.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[col] = df.groupby(['route_id', 'direction', 'day', 'full_hour'])[col].transform(


Total_Passengers missing after fill: 1294


C:\Users\shaha\AppData\Local\Temp\ipykernel_13576\2633105291.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[col] = df.groupby(['route_id', 'direction', 'day'])[col].transform(


Total_Passengers missing after step 2: 683
Total_Passengers missing after step 3: 652
Total_Passengers missing after step 4: 0


,missing_count,missing_percent
Avg_Passengers_Per_Bus,1294,2.340345
line_num,126,0.227885
duration_min_actual,78,0.141072
duration_difference_min,78,0.141072
speed_kmh_actual,78,0.141072
curvity,24,0.043407
perc_within_pt_route,24,0.043407
route_length,24,0.043407
length_in_buffer_m,24,0.043407
gtfs_route_id,0,0.000000


### 2.3 Duration columns

In [8]:
### Impute the remaining nulls of the duration and speed based on agency, line and hor average

df = impute_by_agency_line_hour(df, 'duration_min_actual')
df = impute_by_agency_line_hour(df, 'duration_difference_min')
df = impute_by_agency_line_hour(df, 'speed_kmh_actual')

duration_min_actual: filled 78 values using 'mean'
duration_difference_min: filled 78 values using 'mean'
speed_kmh_actual: filled 78 values using 'mean'


### 2.3. Spatial columns

In [11]:
cols = [
    "curvity",
    "perc_within_pt_route",
    "route_length",
    "length_in_buffer_m"
]

for col in cols:
    df[col] = df[col].fillna(df[col].mean())

### 2.4. Missing values summary

In [12]:
missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percent': df.isnull().mean() * 100
})

missing_summary = missing_summary.sort_values(by='missing_percent', ascending=False)

missing_summary

,missing_count,missing_percent
Avg_Passengers_Per_Bus,1294,2.340345
line_num,126,0.227885
SIRI_id,0,0.000000
duration_min_actual,0,0.000000
duration_difference_min,0,0.000000
speed_kmh_planned,0,0.000000
speed_kmh_actual,0,0.000000
gtfs_route_id,0,0.000000
gtfs_ride_id,0,0.000000
rainfall_mm,0,0.000000


## 3. Outliers handling

### 3.1. Speed columns

#### 3.1.1. Very high speed

In [ ]:
mask_high = df['speed_kmh_planned'] > 100
print(f"Rows with speed > 100: {mask_high.sum()}")

df.loc[mask_high, 'speed_kmh_planned'] = df.loc[mask_high, 'speed_kmh_planned'] / 1000

print(df['speed_kmh_planned'].describe())

#### 3.1.2. Very low speed

In [ ]:
df[df['speed_kmh_planned'] < 0][['date', 'route_length', 'duration_min_planned', 'departure_time_planned', 'arrival_time_planned', 'speed_kmh_planned']].head(10)

## It looks like it happend because the ries ends in the next day

In [ ]:
# זהה נסיעות שמסתיימות אחרי חצות
mask_midnight = df['arrival_time_planned'] < df['departure_time_planned']
print(f"Trips ending after midnight: {mask_midnight.sum()}")

# תקן את duration_min_planned
df.loc[mask_midnight, 'duration_min_planned'] = (
    (df.loc[mask_midnight, 'arrival_time_planned'] + pd.Timedelta(days=1)) - 
    df.loc[mask_midnight, 'departure_time_planned']
).dt.total_seconds() / 60

# תקן את speed_kmh_planned
df.loc[mask_midnight, 'speed_kmh_planned'] = (
    (df.loc[mask_midnight, 'route_length'] / 1000) / 
    (df.loc[mask_midnight, 'duration_min_planned'] / 60)
)

(df[mask_midnight][['departure_time_planned', 'arrival_time_planned', 
                          'duration_min_planned', 'speed_kmh_planned']].head())

In [ ]:
df[df['speed_kmh_actual'] > 500][['route_length', 'duration_min_planned', 'departure_time_planned', 'arrival_time_planned', 'speed_kmh_planned', 'duration_min_actual', 'speed_kmh_actual']].head(10)

In [ ]:
before = len(df)

df = df[(df['duration_difference_min'] >= -120) & 
        (df['duration_difference_min'] <= 120)]

after = len(df)
print(f"Rows removed: {before - after:,} ({(before-after)/before*100:.2f}%)")
print(f"Rows remaining: {after:,}")
print(df['duration_difference_min'].describe().round(2))

#### 3.1.3. Check speed box plot

## 4. Feature selection

In [ ]:
df = df.drop(columns=['route_dir_alt_day_hr', 'line_num_agency_alter_dir', 'SIRI_id', 'gtfs_ride_id', 'gtfs_route_id' ], errors='ignore')
df.dtypes

## 5. Encoding

### 5.1. Ordinal

In [ ]:
day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

day_mapping = {day: i for i, day in enumerate(day_order)}
df['day_encoded'] = df['day'].map(day_mapping)

print(df[['day', 'day_encoded']].drop_duplicates().sort_values('day_encoded'))

### 5.2. Get dummies

In [ ]:
alternative_dummies = pd.get_dummies(df['alternative'], prefix='alternative')

df = pd.concat([df, alternative_dummies], axis=1)

print(f"New columns added: {alternative_dummies.columns.tolist()}")
print(df[['alternative'] + alternative_dummies.columns.tolist()].drop_duplicates().sort_values('alternative'))

### 5.3. Target

In [ ]:


target_cols = ['agency_name', 'origin_city', 'destination_city', 
               'origin_station', 'destination_station']

te = TargetEncoder()
df[[f'{col}_encoded' for col in target_cols]] = te.fit_transform(
    df[target_cols], 
    df['duration_difference_min']
)

print("New encoded columns:")
for col in target_cols:
    print(f"\n{col} vs {col}_encoded:")
    print(df[[col, f'{col}_encoded']].drop_duplicates().head(5).round(2))

## 6. Feature engieneering

In [ ]:
### Add circular route flag
gdf_trips = add_circular_flag(gdf_trips, threshold=500)

In [ ]:
gdf_trips['urban'] = gdf_trips['route_length'] <=25000

In [ ]:
### Calculating the percetnage of the distance inside the public transport path from the entire route length
gdf_trips['perc_within_pt_route'] = round(gdf_trips['length_in_buffer_m']/gdf_trips['route_length']*100,2)

In [ ]:
peak_hours = [7, 8, 9, 14, 15, 16, 17]

df['perc_within_pt_route_peak'] = df.apply(
    lambda row: row['perc_within_pt_route'] if row['full_hour'] in peak_hours else 0,
    axis=1
)

df['is_peak_hour'] = df['full_hour'].isin(peak_hours).astype(int)

## 7. Manipulation pipelines